## Steam Game Ranking Pipeline 전체 흐름도

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                        Bronze → Silver → Gold Pipeline                           │
└─────────────────────────────────────────────────────────────────────────────────┘

📁 Source (CSV)                    🥉 Bronze                      🥈 Silver                       🥇 Gold
───────────────                    ────────────                    ────────────                    ────────────
/Volumes/training/                 training.sch.                   training.sch.                   training.sch.
sch/study_volume/                  bronze_games_ranking            silver_games_ranking             gold_games_ranking
games_ranking.csv

       │                                  │                                │                              │
       ▼                                  ▼                                ▼                              ▼
┌──────────────┐                  ┌──────────────────┐           ┌──────────────────┐          ┌──────────────────────┐
│ 원본 CSV      │    read.csv()    │ game_name STRING │  lower()  │ game_name STRING │ groupBy  │ game_name            │
│ 672 rows     │ ──────────────▶  │ genre     STRING │  replace  │ genre     STRING │  + agg   │ genres       [array] │
│ 4 columns    │   schema 적용    │ rank_type STRING │  cast     │ rank_type STRING │ ──────▶  │ rank_types   [array] │
│              │                  │ rank      STRING │           │ rank      INT    │          │ rank_type_count      │
└──────────────┘                  └──────────────────┘           └──────────────────┘          │ best_rank            │
                                                                                               │ avg_rank             │
                                                                                               │ top_10_count         │
                                                                                               │ top_100_count        │
                                                                                               │ top_10_rate          │
                                                                                               └──────────────────────┘
                                                                                                303 games (집계)
```

### 각 레이어별 역할

| 레이어 | 노트북 | 주요 작업 | 출력 테이블 |
|--------|--------|-----------|-------------|
| **Bronze** | `bronze_game_data` | CSV 로드, 스키마 정의, 원본 그대로 저장 | `training.sch.bronze_games_ranking` |
| **Silver** | `silver_game_data` | `game_name` 소문자 + 공백→언더스코어, `rank` INT 변환 | `training.sch.silver_games_ranking` |
| **Gold** | `gold_game_data` | 중복 제거, null 필터링, 게임 단위 집계 | `training.sch.gold_games_ranking` |

### Gold 레이어 변환 상세
1. **중복 제거**: `dropDuplicates(["game_name", "genre", "rank_type"])`
2. **Null 방어**: `filter(rank.isNotNull())`
3. **게임 단위 집계**: `groupBy("game_name")` → 장르 목록, 랭킹 카테고리 수, 최고/평균 순위, Top 10/100 진입 횟수
4. **정규화 지표**: `top_10_rate` = Top 10 진입 횟수 ÷ 등장 카테고리 수
5. **정렬**: 카테고리 수 DESC → Top 10 횟수 DESC → 평균 순위 ASC → 게임명 ASC

In [0]:
# define var, lib
import pyspark.sql.functions as F
from pyspark.sql.window import Window

silver_table = 'training.sch.silver_games_ranking'
gold_table = 'training.sch.gold_games_ranking'

In [0]:
silver_df = spark.table(silver_table)

silver_df.display()

game_name,genre,rank_type,rank
counter-strike_2,Action,Sales,1
"warhammer_40,000:_space_marine_2",Action,Sales,2
cyberpunk_2077,Action,Sales,3
black_myth:_wukong,Action,Sales,4
elden_ring,Action,Sales,5
pubg:_battlegrounds,Action,Sales,6
dragon_ball:_sparking!_zero,Action,Sales,7
apex_legends™,Action,Sales,8
dota_2,Action,Sales,9
party_animals,Action,Sales,10


In [0]:
# 1. 중복 제거 및 null rank 필터링
clean_df = (
    silver_df
    .dropDuplicates(["game_name", "genre", "rank_type"])
    .filter(F.col("rank").isNotNull())
)

# 2. 게임 단위 Gold 집계
gold_game_df = (
    clean_df
    .groupBy("game_name")
    .agg(
        F.collect_set("genre").alias("genres"),
        F.collect_set("rank_type").alias("rank_types"),
        F.countDistinct("rank_type").alias("rank_type_count"),
        F.min("rank").alias("best_rank"),
        F.round(F.avg("rank"), 2).alias("avg_rank"),
        F.sum(
            F.when(F.col("rank") <= 10, 1).otherwise(0)
        ).alias("top_10_count"),
        F.sum(
            F.when(F.col("rank") <= 100, 1).otherwise(0)
        ).alias("top_100_count")
    )
    .withColumn(
        "top_10_rate",
        F.round(F.col("top_10_count") / F.col("rank_type_count"), 2)
    )
    .orderBy(
        F.col("rank_type_count").desc_nulls_last(),
        F.col("top_10_count").desc_nulls_last(),
        F.col("avg_rank").asc_nulls_last(),
        F.col("game_name").asc()
    )
)

gold_game_df.display()

game_name,genres,rank_types,rank_type_count,best_rank,avg_rank,top_10_count,top_100_count,top_10_rate
black_myth:_wukong,"List(Action, Adventure, Role-Playing)","List(Sales, Revenue, Review)",3,2,3.0,9,9,3.0
dota_2,"List(Action, Role-Playing, Strategy)","List(Sales, Revenue, Review)",3,1,4.56,9,9,3.0
baldur's_gate_3,"List(Adventure, Role-Playing)","List(Sales, Revenue, Review)",3,3,5.17,6,6,2.0
crusader_kings_iii,"List(Role-Playing, Simulation, Strategy)","List(Sales, Revenue, Review)",3,1,7.0,6,7,2.0
persona_3_reload,"List(Adventure, Role-Playing, Strategy)","List(Sales, Revenue, Review)",3,1,9.86,6,7,2.0
elden_ring,"List(Action, Role-Playing)","List(Sales, Revenue, Review)",3,3,5.67,5,6,1.67
cyberpunk_2077,"List(Action, Role-Playing)","List(Sales, Revenue, Review)",3,1,7.0,5,6,1.67
grand_theft_auto_v,"List(Action, Adventure)","List(Sales, Revenue, Review)",3,2,7.67,5,6,1.67
satisfactory,"List(Adventure, Simulation)","List(Sales, Revenue, Review)",3,3,5.6,4,5,1.33
persona_5_royal,"List(Role-Playing, Strategy)","List(Sales, Revenue, Review)",3,6,15.17,4,6,1.33


In [0]:
# Gold 테이블로 저장 (overwrite)
gold_game_df.write.mode("overwrite").saveAsTable(gold_table)

print(f"Gold 테이블 저장 완료: {gold_table}")
print(f"저장된 행 수: {spark.table(gold_table).count()}")

Gold 테이블 저장 완료: training.sch.gold_games_ranking
저장된 행 수: 303


In [0]:
# 게임별 rank_type 내 최고 순위 (장르 무관)
best_per_game = (
    clean_df
    .groupBy("game_name", "rank_type")
    .agg(F.min("rank").alias("best_rank"))
)

# rank_type별 상위 5개 게임
window_spec = Window.partitionBy("rank_type").orderBy(F.col("best_rank").asc())

top5_by_rank_type = (
    best_per_game
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") <= 5)
    .select("rank_type", "row_num", "game_name", "best_rank")
    .orderBy("rank_type", "row_num")
)

top5_by_rank_type.display()

rank_type,row_num,game_name,best_rank
Revenue,1,ea_sports_fc™_25,1
Revenue,2,persona_3_reload,1
Revenue,3,"warhammer_40,000:_space_marine_2",1
Revenue,4,crusader_kings_iii,1
Revenue,5,black_myth:_wukong,2
Review,1,terraria,1
Review,2,dota_2,1
Review,3,counter-strike_2,1
Review,4,garry's_mod,1
Review,5,all-in-one_sports_vr,1


Databricks visualization. Run in Databricks to view.